In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pycountry

GOOGLE = "data/raw/google-government-removal-requests.csv"
PANEL = "data/country_panel_wdi_who.csv"

print("ready")

ready


In [2]:
g = pd.read_csv(GOOGLE, keep_default_na=False)
print(g.shape)
g.head()

(27991, 14)


,time_period,country_name,cldr_territory,requestor,product,reason,number_of_requests,items_requested,items_removed_legal,items_removed_policy,items_content_not_found,items_not_enough_info,items_no_action_taken,items_content_already_removed
0,January - June 2011,Argentina,AR,Executive,Adwords,Defamation,1,4,4,0,0,0,0,0
1,January - June 2011,Argentina,AR,Judicial,Blogger,Defamation,4,5,5,0,0,0,0,0
2,January - June 2011,Argentina,AR,Judicial,Blogger,Privacy and security,1,1,1,0,0,0,0,0
3,January - June 2011,Argentina,AR,Judicial,Web Search,Defamation,6,34,34,0,0,0,0,0
4,January - June 2011,Argentina,AR,Judicial,Web Search,Other,1,1,1,0,0,0,0,0


In [3]:
num = ["number_of_requests", "items_requested", "items_removed_legal", "items_removed_policy"]
for col in num:
    g[col] = pd.to_numeric(g[col], errors="coerce").fillna(0)

g["removed"] = g.items_removed_legal + g.items_removed_policy
g["year"] = g.time_period.str[-4:].astype(int)
g = g[g.year >= 2019].copy()

print(g.shape, g.year.min(), g.year.max())

(19417, 16) 2019 2025


In [4]:
c = (g.groupby(["country_name", "cldr_territory"])
       .agg(req=("number_of_requests", "sum"),
            items=("items_requested", "sum"),
            removed=("removed", "sum"))
       .reset_index())

c["share_removed"] = c.removed / c["items"]
c = c[c["items"] >= 100].copy()

print(len(c))
c[c.country_name == "Kenya"]

84


,country_name,cldr_territory,req,items,removed,share_removed
75,Kenya,KE,93,271,45,0.166052


In [5]:
AFR = set("DZ AO BJ BW BF BI CM CV CF TD KM CG CD CI DJ EG GQ ER SZ ET GA GM GH GN GW KE LS LR LY MG MW ML MR MU MA MZ NA NE NG RW ST SN SC SL SO ZA SS SD TZ TG TN UG ZM ZW".split())

c["africa"] = c.cldr_territory.isin(AFR)
print(c.africa.sum())
c[c.africa].sort_values("share_removed")[["country_name", "items", "share_removed"]]

9


,country_name,items,share_removed
139,Tanzania,126,0.111111
75,Kenya,271,0.166052
33,Cote dIvoire,100,0.210000
103,Nigeria,348,0.261494
130,South Africa,127,0.299213
44,Egypt,262,0.385496
2,Algeria,140,0.692857
124,Senegal,100,0.950000
16,Benin,1998,1.000000


In [6]:
p = pd.read_csv(PANEL)
p = p[(p.year >= 2019) & (p.year <= 2024)]

ctx = (p.groupby("country_code")
         .agg(voice=("voice_accountability_index", "mean"),
              income=("income_group", "last"))
         .reset_index())

c["iso3"] = c.cldr_territory.map(
    lambda x: getattr(pycountry.countries.get(alpha_2=x), "alpha_3", None) if x else None)
c = c.merge(ctx, how="left", left_on="iso3", right_on="country_code")
c["income"] = c.income.fillna("No income data")

print(c.income.value_counts())
c[c.country_name == "Kenya"][["country_name", "income", "voice"]]

income
High income            42
Upper middle income    26
Lower middle income    14
No income data          2
Name: count, dtype: int64


,country_name,income,voice
45,Kenya,Lower middle income,-0.374956


In [7]:
med = c.share_removed.median()
q1, q3 = c.share_removed.quantile([.25, .75])
ken = c[c.country_name == "Kenya"].iloc[0]
rank_low = int(c.share_removed.rank().loc[c.country_name == "Kenya"].iloc[0])

c["country_name"] = c.country_name.replace({"Cote dIvoire": "Côte d'Ivoire"})
c["hover"] = [
    f"<b>{r.country_name}</b><br>Items requested: {int(r['items']):,}"
    f"<br>Share removed: {r.share_removed:.0%} ({int(r.removed):,} items)"
    f"<br>Requests: {int(r.req):,}<br>Income group: {r.income}"
    f"<br>Voice & accountability: " + (f"{r.voice:+.2f}" if pd.notna(r.voice) else "n/a")
    for _, r in c.iterrows()]

print(f"median {med:.1%} | middle half {q1:.0%} to {q3:.0%} | Kenya rank {rank_low}")

median 56.6% | middle half 29% to 78% | Kenya rank 15


In [13]:
GREY, ORANGE, BLUE, MUTED = "#B4B4B4", "#E69F00", "#0072B2", "#6B6B6B"

others = c[c.country_name != "Kenya"]
world = others[~others.africa]
africa = others[others.africa]
kenya = c[c.country_name == "Kenya"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=world["items"], y=world.share_removed, mode="markers", name="Rest of world",
                         hovertext=world.hover, hoverinfo="text",
                         marker=dict(size=11, color=GREY, opacity=.85, line=dict(color="white", width=1))))
fig.add_trace(go.Scatter(x=africa["items"], y=africa.share_removed, mode="markers+text", name="Other African countries",
                         text=africa.country_name, textposition="top right", textfont=dict(size=12),
                         hovertext=africa.hover, hoverinfo="text",
                         marker=dict(size=14, color=ORANGE, line=dict(color="white", width=1.2))))
fig.add_trace(go.Scatter(x=kenya["items"], y=kenya.share_removed, mode="markers+text", name="Kenya",
                         text=["Kenya 17%"], textposition="middle right", textfont=dict(size=15, color=BLUE),
                         hovertext=kenya.hover, hoverinfo="text",
                         marker=dict(size=22, color=BLUE, line=dict(color="white", width=2))))

fig.update_layout(
    title="Kenya's 17% is low worldwide, and among African countries only Tanzania is lower",
    xaxis=dict(type="log", title="Items governments asked Google to remove, 2019–2025 (log scale)"),
    yaxis=dict(title="Share of items removed", tickformat=".0%", range=[-.04, 1.05]),
    plot_bgcolor="white", height=600)
fig.show()

## Kenya's takedown requests: how often does Google comply?

Since 2019, Google removed only **17%** of the items Kenya asked it to take down (45 of 271). The median country sees **57%**, and Kenya ranks 15th lowest of 84 countries.

Among the nine African countries in the data, only **Tanzania** (11%) is lower. Egypt, Nigeria and South Africa sit between 26% and 39%, while Benin and Senegal approach 100%.